# Istio Ambient, end to end — a petshop walkthrough

**Solo Enterprise for Istio, ambient mode, on kind.** One app (a petshop), every ambient security feature, and the Solo Enterprise differences called out as we go.

What we build, in the order the mesh actually enforces it:

| Step | Layer | What it shows |
|------|-------|---------------|
| Identity = certificate | — | every workload gets a SPIFFE SVID from its ServiceAccount |
| L4 authorization | **L4 · ztunnel** | allow/deny on the peer's identity, no waypoint |
| Access logs | **L4 · ztunnel** | identity-aware allow/deny logs (`src.identity`) |
| The shared-SA gap | **L4** | two pods on one ServiceAccount are indistinguishable |
| A waypoint | **L7 · waypoint** | opt-in Envoy for HTTP concerns |
| An IdP + JWT | — | Keycloak issues tokens (`alice` user, `bob` admin) |
| JWT authorization | **L7 · waypoint** | `RequestAuthentication` + claim-based `AuthorizationPolicy` |
| Solo Enterprise extras | — | L7 telemetry from ztunnel (no waypoint), 1.30 workload-claims |

> **Where JWTs live:** a request JWT is HTTP, so it is validated and authorized at the **waypoint (L7)**. ztunnel at **L4** authorizes on the workload's **certificate identity**, never the user's token. This notebook shows both, and where each belongs.

Every step below shows the real command and the real manifest — nothing hidden behind a wrapper script.


## Prerequisites

- `docker`, `kind`, `kubectl`, `helm`, `istioctl`, `jq`, and an **authenticated `gcloud`** (Solo Istio images pull from `us-docker.pkg.dev`).
- `SOLO_ISTIO_LICENSE_KEY` exported (or `SECRETS_FILE` pointing at a file that exports it).

**Edition:** Enterprise — the mesh is the Solo distribution, installed straight from the **Helm charts** (no operator). Validated on Solo Istio `1.29.3-solo` (Helm charts `1.29.3-solo`), Gateway API `v1.5.1`.


In [ ]:
# Run from the lab directory. Constants used throughout.
[ -d istio-ambient-cert-identity-kind ] && cd istio-ambient-cert-identity-kind || :
export CTX=kind-cert-identity
export ISTIO_NS=istio-system
# license (edit if your secrets live elsewhere)
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "context: $CTX ; license set: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"


## 1 · Stand up the mesh

Ambient here is the **four Solo-distribution Helm charts** (`base`, `istiod`, `cni`, `ztunnel`), profile `ambient` — **no Gloo Operator**. The things an operator would hide (the licence, the trust domain, JSON access logs) are plain Helm **values**, set up front, so there are no post-hoc `kubectl patch` steps. `scripts/setup-cluster.sh` runs exactly these.


In [ ]:
# 1a. kind cluster (1 control-plane + 1 worker)
kind create cluster --config kind/cluster.yaml


In [ ]:
# 1b. Gateway API CRDs (used by the waypoint later)
kubectl --context $CTX apply --server-side -f \
  https://github.com/kubernetes-sigs/gateway-api/releases/download/v1.5.1/standard-install.yaml


In [ ]:
# 1c. Pre-pull the Solo Istio images on the host (gcloud-authenticated) and load them into kind.
#     (docker save | kind load, because kind's containerd chokes on multi-arch indexes.)
REG=us-docker.pkg.dev/soloio-img/istio ; VER=1.29.3
case "$(uname -m)" in arm64|aarch64) PLAT=linux/arm64;; *) PLAT=linux/amd64;; esac
for img in pilot proxyv2 install-cni ztunnel; do
  docker pull -q $REG/$img:$VER
  docker save --platform $PLAT $REG/$img:$VER -o /tmp/$img.tar
  kind load image-archive /tmp/$img.tar --name cert-identity && rm -f /tmp/$img.tar
done


Chart version keeps the `-solo` suffix (`1.29.3-solo`); the image tag does not (`1.29.3`). The trust domain is set directly with `meshConfig.trustDomain: cert-identity`, so identities are `spiffe://cert-identity/ns/<ns>/sa/<sa>` — **not** `cluster.local`.


In [ ]:
# helm repo + versions for the Solo distribution
export HUB=us-docker.pkg.dev/soloio-img/istio TAG=1.29.3
export HREPO=oci://us-docker.pkg.dev/soloio-img/istio-helm HVER=1.29.3-solo

# 1d. base — Istio CRDs + cluster roles
helm --kube-context $CTX upgrade -i istio-base $HREPO/base \
  -n $ISTIO_NS --create-namespace --version $HVER --set defaultRevision=default --wait


In [ ]:
# 1e. istiod (profile ambient) — LICENCE, TRUST DOMAIN and access logs are VALUES, not patches
helm --kube-context $CTX upgrade -i istiod $HREPO/istiod -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
global: { hub: $HUB, tag: $TAG }
istio_cni: { enabled: true }
license: { value: $SOLO_ISTIO_LICENSE_KEY }
env: { PILOT_SKIP_VALIDATE_TRUST_DOMAIN: "true" }
meshConfig:
  accessLogFile: /dev/stdout
  trustDomain: cert-identity
EOF


In [ ]:
# 1f. istio-cni (traffic capture) + ztunnel (per-node L4 proxy).
#     ztunnel gets JSON logs + Solo L7 telemetry as env VALUES — again, no kubectl patch.
helm --kube-context $CTX upgrade -i istio-cni $HREPO/cni -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
global: { hub: $HUB, tag: $TAG }
ambient: { dnsCapture: true }
excludeNamespaces: [istio-system, kube-system]
EOF

helm --kube-context $CTX upgrade -i ztunnel $HREPO/ztunnel -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
hub: $HUB
tag: $TAG
namespace: $ISTIO_NS
istioNamespace: $ISTIO_NS
env: { LOG_FORMAT: json, L7_ENABLED: "true" }
EOF
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel daemonset/istio-cni-node --timeout=180s
echo "ambient ready (Helm, no operator), trust domain cert-identity, JSON access logs on"


## 2 · Watch it in the Solo UI (Gloo UI)

The **Gloo UI** is Solo's own dashboard for Solo Enterprise for Istio. It is the Gloo Platform management plane; on one kind cluster the mgmt server, the agent and the UI are co-located, and once the cluster is registered the agent discovers the ambient mesh. Install it **now, before the petshop**, so you can watch the workloads show up.

Needs `GLOO_PLATFORM_LICENSE_KEY` (falls back to `SOLO_ISTIO_LICENSE_KEY`). This is a management plane — several pods, a few minutes to settle.


In [ ]:
# 2a. Gloo Platform management plane + Gloo UI. Single cluster, so the mgmt
#     server, the agent and the UI are co-located; the agent relays over the
#     in-cluster service. gloo-platform 2.12.x pairs with Istio 1.29.
export GLOO_MESH_NS=gloo-mesh GLOO_PLATFORM_VERSION=2.12.3
helm repo add gloo-platform https://storage.googleapis.com/gloo-platform/helm-charts >/dev/null 2>&1 || true
helm repo update gloo-platform >/dev/null
kubectl --context $CTX create namespace $GLOO_MESH_NS --dry-run=client -o yaml | kubectl --context $CTX apply -f -

helm --kube-context $CTX upgrade -i gloo-platform-crds gloo-platform/gloo-platform-crds \
  -n $GLOO_MESH_NS --version $GLOO_PLATFORM_VERSION --wait

helm --kube-context $CTX upgrade -i gloo-platform gloo-platform/gloo-platform \
  -n $GLOO_MESH_NS --version $GLOO_PLATFORM_VERSION --wait --timeout 10m -f - <<EOF
common:
  cluster: cert-identity
licensing:
  glooMeshLicenseKey: "${GLOO_PLATFORM_LICENSE_KEY:-$SOLO_ISTIO_LICENSE_KEY}"
glooMgmtServer:
  enabled: true
  createGlobalWorkspace: true
glooUi:
  enabled: true
  serviceType: ClusterIP
glooAgent:
  enabled: true
  relay:
    serverAddress: gloo-mesh-mgmt-server.gloo-mesh:9900
prometheus:
  enabled: true
redis:
  deployment:
    enabled: true
telemetryCollector:
  enabled: true
telemetryGateway:
  enabled: true
glooInsightsEngine:
  enabled: true
EOF

# register this cluster, or the co-located agent is rejected ("not registered")
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: admin.gloo.solo.io/v2
kind: KubernetesCluster
metadata: { name: cert-identity, namespace: gloo-mesh }
spec: { clusterDomain: cluster.local }
EOF


Port-forward the UI in the **background** so the cell returns immediately and does not hang:


In [ ]:
# 2b. background port-forward — nohup + & returns straight away; logs to /tmp
nohup kubectl --context $CTX -n gloo-mesh port-forward svc/gloo-mesh-ui 8090:8090 \
  > /tmp/cert-identity-gloo-ui-pf.log 2>&1 &
sleep 4
echo "Gloo UI → http://localhost:8090"


Open **http://localhost:8090**. Deploy the petshop in the next step and refresh: `petstore`, `storefront`, `analytics` and both `checkout` workloads appear under the `petshop` namespace, and once traffic flows the Observability graph fills in (on the Solo distribution ztunnel emits L7 metrics with no waypoint, so you get HTTP edges even at L4).


## 3 · The petshop app

One namespace, enrolled in ambient with a single label. The cast:

| Workload | ServiceAccount → identity | Role |
|---|---|---|
| `petstore` | `sa/petstore` | the API (`GET /pets`, `DELETE /pets/{id}`) |
| `storefront` | `sa/storefront` | client the L4 policy **allows** |
| `analytics` | `sa/analytics` | client the L4 policy **denies** |
| `checkout-blue` | `sa/checkout` | shares one SA with green … |
| `checkout-green` | `sa/checkout` | … so both present the **same** SVID |

**Where the traffic comes from:** each client runs a `curl http://petstore:8080/pets` loop *every 2 seconds* and prints the HTTP code. The observe-cells below don't send requests themselves — they read the client's last log line (`kubectl logs --tail=1`) to see the current outcome, which is why each policy step does a short `sleep` first (to let the new rule reach ztunnel). A denied client shows `000000` (connection refused), an allowed one `200`.


In [ ]:
# Enrol the namespace in ambient — one label, no restarts.
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: petshop
  labels:
    istio.io/dataplane-mode: ambient
EOF

# Deploy the petshop (manifests in yaml/10-app/ — petstore is the API, the rest are clients)
kubectl --context $CTX apply -f yaml/10-app/
kubectl --context $CTX -n petshop rollout status deploy/petstore deploy/storefront deploy/analytics deploy/checkout-blue deploy/checkout-green --timeout=180s
kubectl --context $CTX -n petshop get pods


## 4 · Identity is the certificate

ztunnel holds one mTLS SVID per workload identity. The URI SAN is the ServiceAccount — and that is exactly what an L4 policy matches on. Watch the gap: `checkout-blue` and `checkout-green` share `sa/checkout`, so ztunnel holds **one** cert for both.


In [ ]:
# Which ztunnel is on the node with our pods, then the SVIDs it holds
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
istioctl --context $CTX ztunnel-config certificate "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE NAME|petshop" | grep -Ei "name|leaf"


## 5 · Authorize on identity, at L4

One `AuthorizationPolicy`, enforced by ztunnel, no waypoint. It selects `petstore` and allows only the `storefront` identity — so it simultaneously **allows storefront** and **denies analytics and checkout** (ztunnel fails closed: once any ALLOW selects a workload, everything unnamed is denied).

Principals use the trust domain `cert-identity`, **not** `cluster.local`.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-storefront, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["cert-identity/ns/petshop/sa/storefront"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
# each client curls petstore every 2s and prints the HTTP code
for d in storefront analytics checkout-blue checkout-green; do
  echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"
done


## 6 · Read the L4 access logs

ztunnel logs every connection with the peer SPIFFE identities and the outcome — identity-aware audit at L4, no waypoint.


In [ ]:
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
# a DENY (analytics) and an ALLOW (storefront), projected to the fields that matter
kubectl --context $CTX -n $ISTIO_NS logs "$ZT" --tail=500 | grep petshop \
 | grep -E 'policy rejection|http_access' | tail -4 \
 | jq -rc '{src:(.["src.identity"]//"-"), dst:(.["dst.service"]//"-"), method:(.method//"L4"), code:(.response_code//"-"), error:(.error//"ok")}'


## 7 · The shared-ServiceAccount ceiling

Add `sa/checkout` to the allowed set. Both `checkout-blue` and `checkout-green` get in — and there is **no L4 rule that lets one in and keeps the other out**, because they present the same SVID. This is the limit of SA-scoped identity, and the motivation for §13.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-checkout, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["cert-identity/ns/petshop/sa/checkout"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
for d in storefront analytics checkout-blue checkout-green; do
  echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"
done
# -> storefront 200, checkout-blue 200, checkout-green 200, analytics still 000000


## 8 · More of the L4 match surface — namespace, `when`, and DENY

Every policy so far matched on **identity** (the SPIFFE principal). ztunnel can decide on more than that, still at L4, still with no waypoint. The things it can see about a connection:

- **who** is calling — `principals` (the identity), which is what we've used so far
- **where** the caller is — its source `namespaces` (or source IP)
- **where it's going** — the destination `port`

Two more moves round it out: any of these can be written as a CEL **`when`** clause (not only as a top-level `from`/`to` field), and a **`DENY`** rule always beats an `ALLOW`.

To show a decision based on **where** a caller is (rather than who it is), we add a second client, `warehouse-svc`, in a *different* namespace (`warehouse`), then allow only callers whose namespace is `petshop` — so `warehouse-svc` is refused.


In [ ]:
# A second client, in ANOTHER namespace, so we can decide on WHERE a caller is.
# It has its own identity (spiffe://cert-identity/ns/warehouse/sa/warehouse-svc)
# and calls petstore across namespaces. (Same manifest as yaml/25-l4-surface/00-warehouse.yaml.)
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: warehouse
  labels:
    istio.io/dataplane-mode: ambient
---
apiVersion: v1
kind: ServiceAccount
metadata: { name: warehouse-svc, namespace: warehouse }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: warehouse-svc, namespace: warehouse, labels: { app: warehouse-svc } }
spec:
  replicas: 1
  selector: { matchLabels: { app: warehouse-svc } }
  template:
    metadata: { labels: { app: warehouse-svc } }
    spec:
      serviceAccountName: warehouse-svc
      containers:
        - name: client
          image: curlimages/curl:8.14.1
          command: ["/bin/sh","-c"]
          args:
            - |
              while true; do
                code=$(curl -s -o /dev/null -w '%{http_code}' --max-time 3 http://petstore.petshop:8080/pets || echo 000)
                echo "$(date -u +%H:%M:%S) warehouse-svc -> GET petstore.petshop/pets : $code"
                sleep 2
              done
EOF
kubectl --context $CTX -n warehouse rollout status deploy/warehouse-svc --timeout=90s


**Decide on the source namespace — with a CEL `when` clause.** Allow only callers whose `source.namespace` is `petshop`, so the cross-namespace `warehouse-svc` is denied. (`from.source.namespaces: ["petshop"]` is the equivalent without a `when`; the `when` form generalises to any L4 attribute ztunnel knows.)


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-allow-petshop-namespace, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - to:   [{ operation: { ports: ["8080"] } }]
    when: [{ key: source.namespace, values: ["petshop"] }]
EOF
sleep 14
echo "petshop:   $(kubectl --context $CTX -n petshop   logs deploy/storefront    --tail=1)"
echo "warehouse: $(kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1)"


**DENY beats ALLOW.** ztunnel evaluates CUSTOM → DENY → ALLOW, so a `DENY` rule overrides the namespace `ALLOW`. Here `analytics` is in `petshop` (the ALLOW would admit it) but a `DENY` on its identity blocks it anyway — a clean carve-out with no change to the broader policy.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-deny-analytics, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: DENY
  rules:
  - from: [{ source: { principals: ["cert-identity/ns/petshop/sa/analytics"] } }]
EOF
sleep 14
for d in storefront analytics; do echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"; done
# storefront 200 (still allowed), analytics 000000 (denied: DENY > ALLOW)

# reset before the waypoint section
kubectl --context $CTX -n petshop delete authorizationpolicy l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found


## 9 · Add a waypoint for L7

Everything so far was L4. HTTP concerns — methods, paths, JWTs — need a waypoint: a standalone Envoy you opt into per service. We reset the L4 policies here so the L7 story stands on its own (in production you would keep both, defence in depth).


In [ ]:
# reset L4 policies for a clean L7 demo
kubectl --context $CTX -n petshop delete authorizationpolicy allow-storefront allow-checkout --ignore-not-found

# a waypoint (istio-waypoint GatewayClass), then enrol the petstore Service to use it
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: petstore-waypoint
  namespace: petshop
  labels: { istio.io/waypoint-for: service }
spec:
  gatewayClassName: istio-waypoint
  listeners:
  - name: mesh
    port: 15008
    protocol: HBONE
EOF
kubectl --context $CTX label service petstore -n petshop istio.io/use-waypoint=petstore-waypoint --overwrite
kubectl --context $CTX -n petshop rollout status deploy/petstore-waypoint --timeout=150s


## 10 · An IdP, and a real JWT

Keycloak runs in its own (non-ambient) namespace with a `petshop` realm and two users: **alice** (`role: user`) and **bob** (`role: admin`). We mint a token with the password grant and read its claims — the issuer and `realm_access.roles` are what the waypoint will check.


In [ ]:
kubectl --context $CTX apply -f yaml/40-idp/keycloak.yaml
kubectl --context $CTX -n keycloak rollout status deploy/keycloak --timeout=240s


In [ ]:
# mint + decode a token (run from a petshop pod; it reaches Keycloak cross-namespace)
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
BOB=$(tok bob)
echo "$BOB" | cut -d. -f2 | { read p; python3 - "$p" <<'PY'
import sys,base64,json
p=sys.argv[1]; p+="="*(-len(p)%4)
d=json.loads(base64.urlsafe_b64decode(p))
print("iss:  ", d["iss"]); print("user: ", d["preferred_username"]); print("roles:", d["realm_access"]["roles"])
PY
}


## 11 · Authorize on the JWT, at L7

Two objects on the waypoint. `RequestAuthentication` validates the token against Keycloak's JWKS. `AuthorizationPolicy` then requires a valid token for **any** request and — with a CEL `when` clause over the nested `realm_access.roles` claim — restricts `DELETE` to admins.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: RequestAuthentication
metadata: { name: petshop-jwt, namespace: petshop }
spec:
  targetRefs:
  - { kind: Gateway, group: gateway.networking.k8s.io, name: petstore-waypoint }
  jwtRules:
  - issuer:  "http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop"
    jwksUri: "http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/certs"
    forwardOriginalToken: true
---
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: petstore-jwt-authz, namespace: petshop }
spec:
  targetRefs:
  - { kind: Gateway, group: gateway.networking.k8s.io, name: petstore-waypoint }
  action: ALLOW
  rules:
  - from: [{ source: { requestPrincipals: ["*"] } }]           # any valid token may read
    to:   [{ operation: { methods: ["GET"] } }]
  - from: [{ source: { requestPrincipals: ["*"] } }]           # only admins may delete
    to:   [{ operation: { methods: ["DELETE"] } }]
    when: [{ key: "request.auth.claims[realm_access][roles]", values: ["admin"] }]
EOF
sleep 10


In [ ]:
# the matrix: no token, alice (user), bob (admin)
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
call() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c "$1" ; }
ALICE=$(tok alice); BOB=$(tok bob)
echo "no token    GET  /pets     -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 http://petstore:8080/pets")"
echo "alice       GET  /pets     -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets")"
echo "alice(user) DELETE /pets/1 -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets/1")"
echo "bob(admin)  DELETE /pets/1 -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $BOB' http://petstore:8080/pets/1")"


## 12 · What Solo Enterprise adds

**L7 telemetry from ztunnel, with no waypoint.** On the Solo distribution, ztunnel emits HTTP metrics, access logs and traces itself — which is why the *allowed* L4 line back in §6 already carried `method`/`path`/`response_code`. Upstream OSS ztunnel logs are L4-only.


In [ ]:
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
istioctl --context $CTX ztunnel-config all "$ZT.$ISTIO_NS" -o json | jq '.config.l7Config'


## 13 · Closing the gap — 1.30 workload claims (reference)

§7 hit a wall: two pods on one ServiceAccount are indistinguishable at L4. The Solo **1.30 line** (alpha, Enterprise) closes it — CEL over claims baked into the *certificate*, still at L4, still no waypoint. This lab runs 1.29.3-solo, so this is **reference only** (`yaml/30-reference/workload-claims-1.30.yaml`), not applied here.

```yaml
# 1) turn it on:  ENABLE_WORKLOAD_CLAIMS=true on ztunnel  (per-pod certs)
# 2) annotate the pod — the claim is embedded in its mTLS cert at issuance:
metadata:
  annotations:
    solo.io.security-claims/tier: "gold"     # e.g. checkout-blue=gold, checkout-green=silver
# 3) authorize at L4 on the claim, with CEL:
spec:
  action: ALLOW
  rules:
  - when:
    - key: "source.claims['solo.io.security-claims.tier']"   # '/' -> '.'
      values: ["gold"]
```

The SPIFFE URI never changes; the claim rides alongside it, so ztunnel can now tell `checkout-blue` from `checkout-green` — the exact thing §7 could not do. Note the distinction from §11: **this** CEL is over the workload *certificate*; the JWT CEL in §11 is over the user's *request token* at the waypoint.


## 14 · L4 or L7? the decision

| You want to authorize on … | Layer | Where | Waypoint? |
|---|---|---|---|
| Which **workload** (its identity) is calling | L4 | ztunnel | no |
| Source namespace / IP / destination port | L4 | ztunnel | no |
| Per-workload **cert claims** (1.30) | L4 | ztunnel | no |
| A **user's JWT** and its claims | L7 | waypoint | **yes** |
| HTTP method / path / header | L7 | waypoint | **yes** |

L4 is always on and free; reach for a waypoint the moment the decision is about HTTP or an end-user token.


In [ ]:
# tidy up
kind delete cluster --name cert-identity
